Pengumpulan data

In [40]:
import pandas as pd
import numpy as np

df = pd.read_csv("cause_death.csv")
df.head()

,Cause,Type,Year,Data Redundancy,Total Deaths,Source,Page at Source,Source URL
0,Abrasi Air Laut,Bencana Alam,2012,1,0,Profil Kesehatan Indonesia Tahun 2012,Lampiran 3.38,https://pusdatin.kemkes.go.id/download.php?fil...
1,AIDS,Bencana Non Alam dan Penyakit,2000,1,47,Profil Kesehatan Indonesia Tahun 2004,78,https://pusdatin.kemkes.go.id/download.php?fil...
2,AIDS,Bencana Non Alam dan Penyakit,2001,1,99,Profil Kesehatan Indonesia Tahun 2004,78,https://pusdatin.kemkes.go.id/download.php?fil...
3,AIDS,Bencana Non Alam dan Penyakit,2002,1,100,Profil Kesehatan Indonesia Tahun 2004,78,https://pusdatin.kemkes.go.id/download.php?fil...
4,AIDS,Bencana Non Alam dan Penyakit,2003,1,261,Profil Kesehatan Indonesia Tahun 2010,115,https://pusdatin.kemkes.go.id/download.php?fil...


In [41]:
df.fillna(0)
df.drop_duplicates()

q1 = df["Total Deaths"].quantile(0.25)
q3 = df["Total Deaths"].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr 

df_clean = df[(df["Total Deaths"] >= lower) & (df["Total Deaths"] <= upper)]

print(f"Data sebelum cleaning : {len(df)}")
print(f"Data sesudah cleaning : {len(df_clean)}")

Data sebelum cleaning : 680
Data sesudah cleaning : 552


Uji normalitas shapiro wilk

In [42]:
from scipy import stats

shapiro_test = stats.shapiro(df_clean['Total Deaths'])
print(f"P-Value uji Shapiro-Wilk: {shapiro_test.pvalue}")

P-Value uji Shapiro-Wilk: 4.714403494882831e-34


Uji korelasi pearson

In [43]:
from scipy import stats

correlation, p_value = stats.pearsonr(df_clean['Year'], df_clean['Total Deaths'])

print(f"Korelasi antara Year dan Total Deaths: {correlation}")
print(f"P-value: {p_value}")

Korelasi antara Year dan Total Deaths: -0.10708174138273542
P-value: 0.011822788193798232


In [46]:
import statsmodels.api as sm

# Memisahkan variabel independen (Year) dan dependen (Total Deaths)
X = df['Year']  # variabel independen
y = df['Total Deaths']  # variabel dependen

# Menambahkan konstanta untuk menghitung intercept
X = sm.add_constant(X)

# Membuat model regresi
model = sm.OLS(y, X).fit()

# Menampilkan ringkasan hasil regresi
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:           Total Deaths   R-squared:                       0.002
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     1.177
Date:                Fri, 18 Oct 2024   Prob (F-statistic):              0.278
Time:                        17:42:07   Log-Likelihood:                -7033.7
No. Observations:                 680   AIC:                         1.407e+04
Df Residuals:                     678   BIC:                         1.408e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       1.105e+05   1.01e+05      1.099      0.2

In [3]:


# Membaca data
df = pd.read_csv('/Applications/MAMP/htdocs/local/kodingakademi/Python/python 3/machine learning/statistics/inferential/cause_death.csv')

# 1. Uji Shapiro-Wilk untuk normalitas
shapiro_test = stats.shapiro(df['Total Deaths'])
print("Hasil Uji Shapiro-Wilk:", shapiro_test)

# 2. Uji Non-parametrik (Uji Kruskal-Wallis)
kruskal_result = stats.kruskal(df[df['Type'] == 'Bencana Alam']['Total Deaths'],
                                 df[df['Type'] == 'Bencana Non Alam dan Penyakit']['Total Deaths'])
print("Hasil Uji Kruskal-Wallis:", kruskal_result)

# Post Hoc Test menggunakan Dunn
post_hoc = pg.pairwise_tests(dv='Total Deaths', between='Type', data=df, padjust='bonf')
print("\nHasil Post Hoc Dunn:\n", post_hoc)

# 3. Korelasi Pearson
correlation = df['Year'].corr(df['Total Deaths'])
print("Korelasi antara Year dan Total Deaths:", correlation)

# 4. Transformasi pada Total Deaths
df['Log_Total_Deaths'] = np.log(df['Total Deaths'] + 1)  # Menambahkan 1 untuk menghindari log(0)

# Regresi Linier
# Memisahkan variabel independen (Year) dan dependen (Log_Total_Deaths)
X = df['Year']
y = df['Log_Total_Deaths']

# Menambahkan konstanta untuk intercept
X = sm.add_constant(X)

# Membuat model regresi
model = sm.OLS(y, X).fit()

# Menampilkan ringkasan hasil regresi
print(model.summary())

# Menambahkan uji hipotesis dan interval kepercayaan untuk koefisien
print("\nInterval kepercayaan untuk koefisien regresi:")
print(model.conf_int(alpha=0.05))  # 95% interval kepercayaan

# Hipotesis regresi:
# H0: Tidak ada hubungan antara Year dan Log_Total_Deaths
# H1: Ada hubungan antara Year dan Log_Total_Deaths
p_value = model.pvalues[1]  # P-value untuk variabel independen 'Year'
if p_value < 0.05:
    print("Tolak H0: Ada hubungan signifikan antara Year dan Log_Total_Deaths")
else:
    print("Gagal menolak H0: Tidak ada hubungan signifikan antara Year dan Log_Total_Deaths")

# 5. ANOVA
anova_result = pg.anova(dv='Total Deaths', between='Type', data=df)
print("\nHasil ANOVA:\n", anova_result)

# Hipotesis ANOVA:
# H0: Tidak ada perbedaan signifikan antara rata-rata kelompok
# H1: Ada perbedaan signifikan antara rata-rata kelompok
p_value_anova = anova_result['p-unc'].values[0]
if p_value_anova < 0.05:
    print("Tolak H0: Ada perbedaan signifikan antara kelompok.")
else:
    print("Gagal menolak H0: Tidak ada perbedaan signifikan antara kelompok.")

# Menampilkan interval kepercayaan untuk ANOVA menggunakan efek ukuran
# Hitung effect size (np2) dan interval kepercayaan untuk post hoc
es = pg.compute_effsize(df, dv='Total Deaths', between='Type', eftype='np2')
print("\nEffect size dan interval kepercayaan:\n", es)


Hasil Uji Shapiro-Wilk: ShapiroResult(statistic=np.float64(0.15872269728875976), pvalue=np.float64(1.5986698447598898e-47))
Hasil Uji Kruskal-Wallis: KruskalResult(statistic=np.float64(39.84031244411969), pvalue=np.float64(2.75597743970129e-10))

Hasil Post Hoc Dunn:
   Contrast                              A                              B  \
0     Type                   Bencana Alam  Bencana Non Alam dan Penyakit   
1     Type                   Bencana Alam                 Bencana Sosial   
2     Type  Bencana Non Alam dan Penyakit                 Bencana Sosial   

   Paired  Parametric         T         dof alternative         p-unc  \
0   False        True -0.279849  175.856499   two-sided  7.799221e-01   
1   False        True  1.365215  147.005745   two-sided  1.742711e-01   
2   False        True  5.373907  511.209260   two-sided  1.172894e-07   

         p-corr p-adjust       BF10    hedges  
0  1.000000e+00     bonf      0.108 -0.034739  
1  5.228134e-01     bonf      0.543  

/var/folders/q2/2h4p204s56d3vhv_jx5hmc6m0000gn/T/ipykernel_1836/3200407531.py:53: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  p_value = model.pvalues[1]  # P-value untuk variabel independen 'Year'


TypeError: compute_effsize() got an unexpected keyword argument 'dv'